In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
transactions = pd.read_csv("D:\Dataset\Internship Fraud Detection\paysim_transactions_cleaned.csv")
customers = pd.read_csv("D:\Dataset\Internship Fraud Detection\customers_cleaned.csv")

In [3]:
transactions

,transaction_id,customer_id,amount,timestamp,hour,day_of_week,location,merchant_category,is_new_device,device_id,ip_address,is_fraud,transaction_type,card_present,amount_log,is_weekend
0,TX_00000000,CUST_005287,807.91,2026-08-29 11:13:27.372050,16,5,US,Food,False,DEV_4390,48.164.21.194,0,wire,True,6.695688,1
1,TX_00000001,CUST_002180,734.39,2026-08-22 01:00:27.372684,7,4,UK,Retail,False,DEV_4068,212.254.99.120,0,online,False,6.600401,0
2,TX_00000002,CUST_006491,2080.84,2026-08-22 04:53:27.372963,21,3,US,E-commerce,False,DEV_1115,137.60.196.229,0,in-store,False,7.641007,0
3,TX_00000003,CUST_008714,2521.49,2026-09-02 18:32:27.373252,21,3,UK,E-commerce,False,DEV_2305,119.126.69.84,0,atm,True,7.833002,0
4,TX_00000004,CUST_004620,1802.67,2026-08-30 00:06:27.373504,10,4,IN,Food,False,DEV_4547,60.246.141.253,0,wire,False,7.497579,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99101,TX_00099995,CUST_005372,1099.59,2026-08-30 09:54:51.056494,15,4,SG,Travel,False,DEV_3145,133.177.235.124,0,atm,True,7.003602,0
99102,TX_00099996,CUST_003177,3642.76,2026-08-31 07:21:51.056700,7,2,UK,Retail,False,DEV_1804,81.53.199.88,0,in-store,True,8.200771,0
99103,TX_00099997,CUST_001933,1668.13,2026-08-30 20:11:51.056905,16,2,UK,Retail,False,DEV_1074,231.199.249.141,0,in-store,True,7.420058,0
99104,TX_00099998,CUST_006422,1370.69,2026-08-22 16:37:51.057122,6,2,UK,Services,False,DEV_0259,60.121.17.93,0,atm,True,7.223799,0


In [5]:
def create_features(transactions_df, customers_df):
    """Create engineered features for fraud detection"""
    df = transactions_df.copy()
    cust = customers_df.copy()
    
    # Merge customer data
    df = df.merge(cust, on='customer_id', how='left')
    
    # Time-based features
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['transaction_hour'] = df['timestamp'].dt.hour
    df['transaction_day'] = df['timestamp'].dt.day
    df['transaction_month'] = df['timestamp'].dt.month
    df['transaction_weekday'] = df['timestamp'].dt.weekday
    
    # Customer-based features
    df['customer_transaction_count'] = df.groupby('customer_id')['transaction_id'].transform('count')
    df['customer_avg_amount'] = df.groupby('customer_id')['amount'].transform('mean')
    df['customer_std_amount'] = df.groupby('customer_id')['amount'].transform('std')
    df['customer_max_amount'] = df.groupby('customer_id')['amount'].transform('max')
    
    # Rolling features (last 5 transactions per customer)
    df['rolling_avg_amount'] = df.groupby('customer_id')['amount'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )
    df['rolling_std_amount'] = df.groupby('customer_id')['amount'].transform(
        lambda x: x.rolling(5, min_periods=1).std()
    )
    
    # Amount ratio features
    df['amount_ratio_to_avg'] = df['amount'] / df['customer_avg_amount']
    df['amount_ratio_to_max'] = df['amount'] / df['customer_max_amount']
    
    # Device features
    df['device_fraud_count'] = df.groupby('device_id')['is_fraud'].transform('sum')
    df['device_total_count'] = df.groupby('device_id')['transaction_id'].transform('count')
    df['device_fraud_rate'] = df['device_fraud_count'] / df['device_total_count']
    
    # Location features
    df['location_fraud_count'] = df.groupby('location_x')['is_fraud'].transform('sum')
    df['location_total_count'] = df.groupby('location_x')['transaction_id'].transform('count')
    df['location_fraud_rate'] = df['location_fraud_count'] / df['location_total_count']
    
    # Distance from typical location (if known)
    # For now, use location mismatch indicator
    df['location_mismatch'] = (df['location_x'] != df['location_y']).astype(int)
    
    # Risk score interaction
    df['risk_score_amount'] = df['risk_score'] * df['amount']
    
    # Fill NaN values
    df = df.fillna(0)
    
    # Categorical encoding
    categorical_cols = ['segment', 'merchant_category', 'transaction_type', 
                       'location_x', 'location_y']
    
    for col in categorical_cols:
        le = LabelEncoder()
        df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
    
    # Drop original categorical columns and timestamp
    df = df.drop(columns=['timestamp', 'customer_id', 'transaction_id', 
                          'device_id', 'ip_address'] + categorical_cols)
    
    # Drop columns with all zeros or single value
    df = df.loc[:, df.nunique() > 1]
    
    return df


In [6]:
feature_df = create_features(transactions, customers)
print(f"Feature shape: {feature_df.shape}")
print(f"Features: {feature_df.columns.tolist()}")

Feature shape: (99106, 40)
Features: ['amount', 'hour', 'day_of_week', 'is_new_device', 'is_fraud', 'card_present', 'amount_log', 'is_weekend', 'account_age_days', 'risk_score', 'email_verified', 'phone_verified', 'avg_monthly_transactions', 'avg_transaction_amount', 'device_count', 'transaction_hour', 'transaction_day', 'transaction_month', 'transaction_weekday', 'customer_transaction_count', 'customer_avg_amount', 'customer_std_amount', 'customer_max_amount', 'rolling_avg_amount', 'rolling_std_amount', 'amount_ratio_to_avg', 'amount_ratio_to_max', 'device_fraud_count', 'device_total_count', 'device_fraud_rate', 'location_fraud_count', 'location_total_count', 'location_fraud_rate', 'location_mismatch', 'risk_score_amount', 'segment_encoded', 'merchant_category_encoded', 'transaction_type_encoded', 'location_x_encoded', 'location_y_encoded']


In [7]:
feature_df.to_csv('D:\Dataset\Internship Fraud Detection\\features_engineered.csv', index=False)

In [8]:
X = feature_df.drop('is_fraud', axis=1)
y = feature_df['is_fraud']

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")


X shape: (99106, 39)
y shape: (99106,)


In [10]:
print(X.head(10))

    amount  hour  day_of_week  is_new_device  card_present  amount_log  \
0   807.91    16            5          False          True    6.695688   
1   734.39     7            4          False         False    6.600401   
2  2080.84    21            3          False         False    7.641007   
3  2521.49    21            3          False          True    7.833002   
4  1802.67    10            4          False         False    7.497579   
5    90.03     7            1          False          True    4.511189   
6   260.11    14            0          False         False    5.564942   
7  1872.15    20            5          False         False    7.535377   
8  1927.93    11            2          False          True    7.564721   
9   273.10    12            4          False          True    5.613493   

   is_weekend  account_age_days  risk_score  email_verified  ...  \
0           1              1394    0.520467            True  ...   
1           0               256    0.592291      

In [ ]:
tr